# Logistic Regression	Sentence embeddings (MiniLM or similar) + engineered metadata features (frequency/length/party/subject)	Nested CV with randomized search on C and class_weight, plus isotonic calibration and threshold optimization

## Data Loading and Experiment Settings

This cell loads the training data and defines the preprocessing switches that control how the one-step feature pipeline behaves.

The `df` guard matters in notebooks because cells can be re-run out of order. If `df` already exists but is no longer a DataFrame, the notebook reloads the source CSV.

In [6]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import wandb
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    classification_report,
    confusion_matrix,
    balanced_accuracy_score,
    f1_score,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import LinearSVC

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'data' / 'train.csv').exists() and (candidate / 'src').exists():
            return candidate
    raise FileNotFoundError('Could not locate the project root from the current working directory.')

project_root = find_project_root(Path.cwd())
src_path = project_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

from preprocessing.one_step import OneStepOptions, preprocess_one_step
from submit.save_model import save_model

print(f'Project root: {project_root}')

df = None

if 'df' not in globals() or not isinstance(df, pd.DataFrame):
    df = pd.read_csv(project_root / 'data' / 'train.csv')

print(type(df))
print(df.shape)
df.head()

Project root: c:\Users\CmdrC\Documents\MStanwood\truth-classifier-nlp
<class 'pandas.DataFrame'>
(8950, 8)


,id,label,statement,subject,speaker,speaker_job,state_info,party_affiliation
0,81f884c64a7,1,China is in the South China Sea and (building)...,"china,foreign-policy,military",donald-trump,President-Elect,New York,republican
1,30c2723a188,0,With the resources it takes to execute just ov...,health-care,chris-dodd,U.S. senator,Connecticut,democrat
2,6936b216e5d,0,The (Wisconsin) governor has proposed tax give...,"corporations,pundits,taxes,abc-news-week",donna-brazile,Political commentator,"Washington, D.C.",democrat
3,b5cd9195738,1,Says her representation of an ex-boyfriend who...,"candidates-biography,children,ethics,families,...",rebecca-bradley,NaN,NaN,none
4,84f8dac7737,0,At protests in Wisconsin against proposed coll...,"health-care,labor,state-budget",republican-party-wisconsin,NaN,Wisconsin,republican


## Feature Construction

The one-step preprocessing function turns the raw claim data into a modeling table. This is the right place to inspect feature count, row count, and whether the target column is present before building splits.

In [11]:
# ============================================================
# LABEL OPTIONS  (src/preprocessing/label.py)
# Always produces: label (binary 0/1 column)
# ============================================================

# 'skip' -- keep the label column as-is (for training)
# 'drop' -- remove the label column (for inference / test set)
label_option = 'skip'
label_source_col = 'label'


# ============================================================
# ID OPTIONS  (src/preprocessing/id.py)
# ============================================================

# 'drop'        -- remove the ID column entirely
# 'hash_bucket' -- replace IDs with a hash-bucket integer
id_option = 'drop'


# ============================================================
# SUBJECT OPTIONS  (src/preprocessing/subject.py)
# Always produces: subject_clean
# ============================================================

subject_source_col = 'subject'
subject_keep_original = False               # keep raw subject column
subject_clean_text = True                   # lowercase + strip + collapse whitespace
subject_normalize_separators = True         # normalize | ; , & to single space
subject_split_topics = True                 # split multi-topic strings into a list
subject_primary_strategy = 'first'          # 'first' | 'most_frequent' -- pick primary topic
subject_rare_threshold = 10                 # subjects with < N occurrences are 'rare'
subject_rare_label = 'other'               # label applied to grouped rare subjects
subject_group_rare = False                  # requires add_primary + add_grouped_primary
subject_max_topics_for_primary = None       # cap number of topics considered
subject_multi_topic_label = 'multi-topic'  # label when > 1 topic and not capped

# Additional subject feature columns (all False by default)
subject_add_length_features = False         # subject_length, subject_token_count
subject_add_topic_list = False              # subject_topics -- list of individual topics
subject_add_topic_count = False             # subject_topic_count -- number of topics
subject_add_multiple_topics_flag = False    # subject_has_multiple_topics -- 0/1 flag
subject_add_primary = False                 # subject_primary -- single primary topic string
subject_add_grouped_primary = False         # subject_grouped -- primary with rares collapsed; requires add_primary
subject_add_subject_frequency = False       # subject_frequency -- count of primary topic; requires add_primary
subject_add_subject_is_rare = False         # subject_is_rare -- 0/1 flag; requires add_primary
subject_add_subject_primary_true_rate = False  # subject_primary_true_rate -- leakage risk -- CV folds only
subject_label_col = None                    # label column name; required for primary_true_rate

# 'none' | 'standardize' (z-score) | 'normalize' (min-max)
# Scales: topic_count, length, token_count, frequency. NOT binary flags.
subject_scale = 'none'
subject_verbose = False


# ============================================================
# STATEMENT OPTIONS  (src/preprocessing/statement_ds.py)
# Always produces: statement_original, statement_clean
# ============================================================

statement_source_col = 'statement'
statement_original_output_col = 'statement_original'  # always produced; holds pre-clean text
statement_output_col = 'statement_clean'
statement_keep_original = False             # keep the raw source column

# Text normalization
statement_lower = True                      # convert to lowercase
statement_remove_html = True                # strip HTML tags
statement_remove_urls = True                # remove http:// and www. links
statement_replace_numbers = False           # replace numeric tokens with statement_number_token
statement_number_token = '<NUM>'            # token used when replace_numbers=True
statement_stopword_removal = False          # remove common English stopwords
statement_keep_negations = True             # preserve negation words even when removing stopwords
statement_remove_punctuation = False        # strip all punctuation characters
statement_stemmer = 'none'                  # 'none' | 'porter' | 'snowball' -- requires NLTK
statement_lemmatizer = 'none'               # 'none' | 'wordnet' -- requires NLTK
statement_repair_polluted_statements = True # fix malformed/polluted statement text

# Optional feature columns (all False by default)
statement_add_rare_token_features = False   # rare_token_count, avg_token_freq
statement_rare_token_threshold = 1          # tokens with <= N occurrences are 'rare'
statement_token_freqs = None                # pre-computed token frequency dict
statement_add_spelling_errors = False       # spelling_err_count (approximate)
statement_add_lexical_features = False      # char_len, word_count, upper_ratio, counts, digit_ratio
statement_add_pollution_features = False    # tab_count, newline_count, row_spillover_flag
statement_add_ner_features = False          # entity counts (PERSON, ORG, GPE, DATE, NUM, OTHER)
statement_ner_model = 'en_core_web_sm'      # spaCy model; requires: pip install spacy

# Vectorizer ('none' | 'tfidf' | 'bigram' | 'binary' | 'embeddings')
statement_vectorizer_type = 'none'
statement_vectorizer_max_features = None    # None = no vocab limit
statement_vectorizer_min_df = 1             # minimum document frequency
statement_vectorizer_max_df = 1.0           # maximum document frequency ratio
statement_embedding_model = 'all-MiniLM-L6-v2'  # sentence-transformers model
statement_fitted_vectorizer = None          # pre-fitted vectorizer for test-set

# 'none' | 'standardize' (z-score) | 'normalize' (min-max)
# Scales: char_len, word_count, ratios, counts (lexical, rare token, spelling, NER).
# NOT binary flags (row_spillover_flag) or vec_* columns (already normalized).
statement_scale = 'none'
statement_verbose = False


# ============================================================
# SPEAKER OPTIONS  (src/preprocessing/speaker.py)
# Always produces: speaker_clean
# ============================================================

speaker_source_col = 'speaker'
speaker_keep_original = False               # keep raw speaker column
speaker_clean_text = True                   # lowercase + strip whitespace
speaker_normalize_separators = True         # normalize separator characters

# Rare-grouping
speaker_group_rare = False                  # requires add_grouped_speaker
speaker_rare_threshold = 5                  # speakers with < N occurrences are 'rare'
speaker_rare_label = 'other'               # label for rare speakers

# Additional speaker feature columns (all False by default)
speaker_add_length_features = False         # speaker_char_len, speaker_token_count
speaker_add_frequency = False               # speaker_frequency, speaker_frequency_pct
speaker_add_is_rare = False                 # speaker_is_rare -- 0/1 flag
speaker_add_grouped_speaker = False         # speaker_grouped -- rare names collapsed; requires group_rare
speaker_add_title_flag = False              # speaker_has_title -- 0/1 (senator, doctor, CEO, etc.)
speaker_add_comma_flag = False              # speaker_has_comma -- 0/1
speaker_add_period_flag = False             # speaker_has_period -- 0/1 (e.g. 'J. Biden')
speaker_add_token_count = False             # speaker_token_count
speaker_add_speaker_primary_true_rate = False  # speaker_primary_true_rate -- leakage risk -- CV folds only
speaker_label_col = None                    # label column; required for primary_true_rate

# 'none' | 'standardize' (z-score) | 'normalize' (min-max)
# Scales: frequency, frequency_pct, char_len, token_count. NOT binary flags.
speaker_scale = 'none'
speaker_verbose = False


# ============================================================
# SPEAKER JOB OPTIONS  (src/preprocessing/speaker_job.py)
# Always produces: speaker_job_clean
# ============================================================

speaker_job_source_col = 'speaker_job'
speaker_job_keep_original = False           # keep raw speaker_job column
speaker_job_clean_text = True               # lowercase + strip + remove HTML/URLs
speaker_job_normalize_separators = True     # normalize | ; , / & to space

# Rare-grouping
speaker_job_group_rare = False              # requires add_grouped_job
speaker_job_rare_threshold = 5              # jobs with < N occurrences are 'rare'
speaker_job_rare_label = 'other'           # label for rare jobs

# Additional speaker-job feature columns (all False by default)
speaker_job_add_length_features = False     # speaker_job_char_len, speaker_job_token_count
speaker_job_add_frequency = False           # speaker_job_frequency, frequency_pct
speaker_job_add_is_rare = False             # speaker_job_is_rare -- 0/1 flag
speaker_job_add_grouped_job = False         # speaker_job_grouped -- rare jobs collapsed; requires group_rare
speaker_job_add_title_flag = False          # speaker_job_has_title -- 0/1 (senator, CEO, professor)
speaker_job_add_comma_flag = False          # speaker_job_has_comma -- 0/1
speaker_job_add_slash_flag = False          # speaker_job_has_slash -- 0/1
speaker_job_add_ampersand_flag = False      # speaker_job_has_ampersand -- 0/1
speaker_job_add_token_count = False         # speaker_job_token_count
speaker_job_add_job_primary_true_rate = False  # speaker_job_primary_true_rate -- leakage risk -- CV folds only
speaker_job_job_label_col = None            # label column; required for primary_true_rate

# 'none' | 'standardize' (z-score) | 'normalize' (min-max)
# Scales: char_len, token_count, frequency, frequency_pct. NOT binary flags.
speaker_job_scale = 'none'
speaker_job_verbose = False


# ============================================================
# PARTY AFFILIATION OPTIONS  (src/preprocessing/party_affiliation.py)
# Always produces: party_affiliation_clean
# ============================================================

party_affiliation_source_col = 'party_affiliation'
party_affiliation_keep_original = False     # keep raw party_affiliation column
party_affiliation_clean_text = True         # lowercase + strip + remove HTML

# Rare-grouping
party_affiliation_group_rare = False        # requires add_grouped_party
party_affiliation_rare_threshold = 5        # parties with < N occurrences are 'rare'
party_affiliation_rare_label = 'other'     # label for rare parties

# Additional party feature columns (all False by default)
party_affiliation_add_length_features = False      # party_affiliation_char_len
party_affiliation_add_frequency = False             # party_affiliation_frequency, frequency_pct
party_affiliation_add_is_rare = False               # party_affiliation_is_rare -- 0/1 flag
party_affiliation_add_grouped_party = False         # party_affiliation_grouped -- rare parties collapsed; requires group_rare
party_affiliation_add_slash_flag = False            # party_affiliation_has_slash -- 0/1
party_affiliation_add_ampersand_flag = False        # party_affiliation_has_ampersand -- 0/1
party_affiliation_add_comma_flag = False            # party_affiliation_has_comma -- 0/1
party_affiliation_add_parentheses_flag = False      # party_affiliation_has_parentheses -- 0/1
party_affiliation_add_token_count = False           # party_affiliation_token_count
party_affiliation_add_is_major_party = False        # party_affiliation_is_major_party -- 0/1 (democrat/republican)
party_affiliation_add_is_institutional = False      # party_affiliation_is_institutional -- 0/1 (govt bodies)
party_affiliation_add_party_primary_true_rate = False  # leakage risk -- CV folds only
party_affiliation_party_label_col = None            # label column; required for primary_true_rate

# 'none' | 'standardize' (z-score) | 'normalize' (min-max)
# Scales: frequency, frequency_pct, char_len, token_count. NOT binary flags.
party_affiliation_scale = 'none'
party_affiliation_verbose = False


# ============================================================
# STATE INFO OPTIONS  (src/preprocessing/state.py)
# Always produces: state_info_clean
# ============================================================

state_source_col = 'state_info'
state_drop = False                          # True -- drop state_info entirely, add no features
state_keep_original = False                 # keep raw state_info column
state_clean_text = True                     # lowercase + strip + remove HTML
state_normalize_state = True                # expand 2-letter state codes to full names

# Rare-grouping
state_group_rare = False                    # requires add_grouped_state
state_rare_threshold = 5                    # states with < N occurrences are 'rare'
state_rare_label = 'other'                 # label for rare states

# Additional state feature columns (all False by default)
state_add_is_us_state = False               # state_info_is_us_state -- 0/1 flag
state_add_frequency = False                 # state_info_frequency, state_info_frequency_pct
state_add_is_rare = False                   # state_info_is_rare -- 0/1 flag
state_add_grouped_state = False             # state_info_grouped -- rare states collapsed; requires group_rare
state_add_length_features = False           # state_info_char_len
state_add_token_count = False               # state_info_token_count
state_add_has_us_words = False              # state_info_has_us_words -- 0/1 ('us', 'usa', 'united states')
state_add_us_region = False                 # state_info_us_region -- 'northeast'|'south'|'midwest'|'west'

# 'none' | 'standardize' (z-score) | 'normalize' (min-max)
# Scales: frequency, frequency_pct, char_len, token_count. NOT binary flags.
state_scale = 'none'
state_verbose = False


In [8]:
# ============================================================
# FEATURE ENGINEERING OPTIONS  (src/preprocessing/feature_engineering.py)
#
# Cross-column features computed after all per-column modules finish.
# All flags default to False. All output columns are prefixed fe_.
#
# Three families:
#   1. Interaction â€” combine two cleaned columns into a joint category key
#   2. Aggregate   â€” per-group statistics (mean label, mean length, etc.)
#   3. Text-style  â€” linguistic signals from the statement text
# ============================================================

# Source column names. These match the *_clean outputs of the preprocessing
# modules and are always present after preprocess_one_step runs.
# Only change these if you renamed the upstream output columns.
fe_statement_col = 'statement_clean'
fe_statement_original_col = 'statement_original'   # used for proper-noun heuristic
fe_speaker_col = 'speaker_clean'
fe_subject_col = 'subject_clean'
fe_party_col = 'party_affiliation_clean'
fe_speaker_job_col = 'speaker_job_clean'
fe_state_col = 'state_info_clean'

# Label column â€” required only for the *_true_rate aggregate features.
# Set to 'label' when enabling those features inside CV folds.
fe_label_col = None


# ------------------------------------------------------------
# 1. INTERACTION FEATURES
# Concatenate two cleaned columns into a joint categorical key.
# Output format: "<value_a>__<value_b>" (double underscore separator).
# Example: "barack obama__health care" for fe_speaker_subject.
# No leakage risk. All depend only on *_clean columns.
# ------------------------------------------------------------

# fe_speaker_subject: do certain speakers make false claims on specific topics?
# Depends on: fe_speaker_col and fe_subject_col
fe_add_speaker_subject = False

# fe_speaker_party: does a speaker's false-claim rate vary by party affiliation?
# Depends on: fe_speaker_col and fe_party_col
fe_add_speaker_party = False

# fe_subject_party: do topics have different credibility patterns by party?
# Depends on: fe_subject_col and fe_party_col
fe_add_subject_party = False

# fe_speaker_job_subject: does a speaker's role interact with topic credibility?
# Depends on: fe_speaker_job_col and fe_subject_col
fe_add_speaker_job_subject = False

# fe_state_party: captures regional political credibility patterns.
# Depends on: fe_state_col and fe_party_col
fe_add_state_party = False

# fe_speaker_len_bucket: speaker Ã— statement length bucket (short/medium/long).
# Checks if certain speakers tend to make longer or shorter claims differently.
# Depends on: fe_speaker_col and fe_statement_col
fe_add_speaker_statement_len_bucket = False
# Word-count boundaries for short/medium/long buckets: short < bins[0], long > bins[1].
fe_statement_len_bins = (50, 150)


# ------------------------------------------------------------
# 2. AGGREGATE FEATURES
#
# *_true_rate features: WARNING â€” leakage risk.
# These compute the mean label per group on the CURRENT dataframe.
# Only enable them when calling preprocess_one_step on a training fold,
# then map the computed means to validation/test rows manually.
# Set fe_label_col='label' when enabling these.
#
# Non-label aggregates (avg_statement_len, avg_punctuation, avg_number_ratio)
# have no leakage risk.
# ------------------------------------------------------------

# fe_speaker_true_rate: mean label (false-claim rate) per speaker.
# WARNING: leakage risk â€” compute ONLY inside CV training folds!
# Requires fe_label_col to be set.
fe_add_speaker_true_rate = False

# fe_subject_true_rate: mean label per subject topic.
# WARNING: leakage risk â€” compute ONLY inside CV training folds!
# Requires fe_label_col to be set.
fe_add_subject_true_rate = False

# fe_party_true_rate: mean label per party affiliation.
# WARNING: leakage risk â€” compute ONLY inside CV training folds!
# Requires fe_label_col to be set.
fe_add_party_true_rate = False

# fe_speaker_avg_statement_len: mean word count of statements per speaker.
# No leakage risk.
# Depends on: fe_speaker_col and fe_statement_col
fe_add_speaker_avg_statement_len = False

# fe_subject_avg_statement_len: mean word count of statements per subject.
# No leakage risk.
# Depends on: fe_subject_col and fe_statement_col
fe_add_subject_avg_statement_len = False

# fe_speaker_avg_punctuation: mean punctuation-character density per speaker.
# High punctuation may indicate excited or informal speaking style.
# No leakage risk.
# Depends on: fe_speaker_col and fe_statement_col
fe_add_speaker_avg_punctuation = False

# fe_speaker_avg_number_ratio: mean digit-character ratio per speaker.
# Speakers who cite many numbers may be more specific (or more misleading).
# No leakage risk.
# Depends on: fe_speaker_col and fe_statement_col
fe_add_speaker_avg_number_ratio = False


# ------------------------------------------------------------
# 3. TEXT-STYLE FEATURES
# All derived from fe_statement_col (statement_clean by default).
# No leakage risk.
# ------------------------------------------------------------

# fe_negation_count: count of negation words (not, never, no, cannot, etc.).
# Fake claims sometimes use more negations to create doubt.
# Depends on: fe_statement_col
fe_add_negation_count = False

# fe_hedge_count: count of hedge/uncertainty words (maybe, perhaps, might, etc.).
# Hedging language may indicate vague or unverifiable claims.
# Depends on: fe_statement_col
fe_add_hedge_count = False

# fe_absolutist_count: count of absolutist/extreme words (always, everyone, etc.).
# Absolutist language correlates with exaggerated or misleading claims.
# Depends on: fe_statement_col
fe_add_absolutist_count = False

# fe_numeral_count: count of digit sequences in the statement.
# Claims with specific numbers are more verifiable (and sometimes more false).
# Depends on: fe_statement_col
fe_add_numeral_count = False

# fe_proper_noun_count: heuristic count of capitalized non-sentence-start words.
# Uses fe_statement_original_col (statement_original) to preserve casing.
# Depends on: fe_statement_original_col
fe_add_proper_noun_count = False

# fe_readability: Flesch Reading Ease approximation (no external library needed).
# Higher = easier to read (~0â€“100 scale).
# Complex statements may signal more nuanced or technical claims.
# Depends on: fe_statement_col
fe_add_readability = False

# fe_sentiment_polarity + fe_sentiment_subjectivity: TextBlob-based sentiment.
#   Polarity:     -1 (very negative) to +1 (very positive)
#   Subjectivity:  0 (objective)     to  1 (subjective)
# Subjective language may correlate with opinion-based or misleading claims.
# Requires: pip install textblob
# Depends on: fe_statement_col
fe_add_sentiment = False

# 'none' | 'standardize' (z-score) | 'normalize' (min-max)
# Scales: avg_statement_len, avg_punctuation, avg_number_ratio, true_rate features,
#   negation/hedge/absolutist/numeral/proper_noun counts, readability, sentiment.
# NOT: interaction string columns (fe_speaker_subject etc.) or fe_speaker_len_bucket.
fe_scale = 'none'

fe_verbose = False

# ============================================================
# Build FeatureEngineeringOptions inline inside OneStepOptions
# (these fe_* variables are passed as fe_* kwargs below)
# ============================================================

In [9]:
# ============================================================
# BUILD OneStepOptions FROM THE VARIABLES ABOVE
# ============================================================

options = OneStepOptions(
    # --- Label ---
    label_option=label_option,
    label_source_col=label_source_col,

    # --- ID ---
    id_option=id_option,

    # --- Subject ---
    subject_source_col=subject_source_col,
    subject_keep_original=subject_keep_original,
    subject_clean_text=subject_clean_text,
    subject_normalize_separators=subject_normalize_separators,
    subject_split_topics=subject_split_topics,
    subject_primary_strategy=subject_primary_strategy,
    subject_max_topics_for_primary=subject_max_topics_for_primary,
    subject_multi_topic_label=subject_multi_topic_label,
    subject_add_length_features=subject_add_length_features,
    subject_add_topic_list=subject_add_topic_list,
    subject_add_topic_count=subject_add_topic_count,
    subject_add_multiple_topics_flag=subject_add_multiple_topics_flag,
    subject_add_primary=subject_add_primary,
    subject_add_grouped_primary=subject_add_grouped_primary,
    subject_group_rare=subject_group_rare,
    subject_rare_threshold=subject_rare_threshold,
    subject_rare_label=subject_rare_label,
    subject_add_subject_frequency=subject_add_subject_frequency,
    subject_add_subject_is_rare=subject_add_subject_is_rare,
    subject_add_subject_primary_true_rate=subject_add_subject_primary_true_rate,
    subject_label_col=subject_label_col,
    subject_scale=subject_scale,
    subject_verbose=subject_verbose,

    # --- Statement ---
    statement_source_col=statement_source_col,
    statement_original_output_col=statement_original_output_col,
    statement_output_col=statement_output_col,
    statement_keep_original=statement_keep_original,
    statement_lower=statement_lower,
    statement_remove_html=statement_remove_html,
    statement_remove_urls=statement_remove_urls,
    statement_replace_numbers=statement_replace_numbers,
    statement_number_token=statement_number_token,
    statement_stopword_removal=statement_stopword_removal,
    statement_keep_negations=statement_keep_negations,
    statement_stemmer=statement_stemmer,
    statement_lemmatizer=statement_lemmatizer,
    statement_remove_punctuation=statement_remove_punctuation,
    statement_repair_polluted_statements=statement_repair_polluted_statements,
    statement_add_rare_token_features=statement_add_rare_token_features,
    statement_rare_token_threshold=statement_rare_token_threshold,
    statement_token_freqs=statement_token_freqs,
    statement_add_spelling_errors=statement_add_spelling_errors,
    statement_add_lexical_features=statement_add_lexical_features,
    statement_add_pollution_features=statement_add_pollution_features,
    statement_verbose=statement_verbose,
    statement_add_ner_features=statement_add_ner_features,
    statement_ner_model=statement_ner_model,
    statement_vectorizer_type=statement_vectorizer_type,
    statement_vectorizer_max_features=statement_vectorizer_max_features,
    statement_vectorizer_min_df=statement_vectorizer_min_df,
    statement_vectorizer_max_df=statement_vectorizer_max_df,
    statement_embedding_model=statement_embedding_model,
    statement_fitted_vectorizer=statement_fitted_vectorizer,
    statement_scale=statement_scale,

    # --- Speaker ---
    speaker_source_col=speaker_source_col,
    speaker_keep_original=speaker_keep_original,
    speaker_clean_text=speaker_clean_text,
    speaker_normalize_separators=speaker_normalize_separators,
    speaker_group_rare=speaker_group_rare,
    speaker_rare_threshold=speaker_rare_threshold,
    speaker_rare_label=speaker_rare_label,
    speaker_add_length_features=speaker_add_length_features,
    speaker_add_frequency=speaker_add_frequency,
    speaker_add_is_rare=speaker_add_is_rare,
    speaker_add_grouped_speaker=speaker_add_grouped_speaker,
    speaker_add_title_flag=speaker_add_title_flag,
    speaker_add_comma_flag=speaker_add_comma_flag,
    speaker_add_period_flag=speaker_add_period_flag,
    speaker_add_token_count=speaker_add_token_count,
    speaker_add_speaker_primary_true_rate=speaker_add_speaker_primary_true_rate,
    speaker_label_col=speaker_label_col,
    speaker_scale=speaker_scale,
    speaker_verbose=speaker_verbose,

    # --- Speaker job ---
    speaker_job_source_col=speaker_job_source_col,
    speaker_job_keep_original=speaker_job_keep_original,
    speaker_job_clean_text=speaker_job_clean_text,
    speaker_job_normalize_separators=speaker_job_normalize_separators,
    speaker_job_group_rare=speaker_job_group_rare,
    speaker_job_rare_threshold=speaker_job_rare_threshold,
    speaker_job_rare_label=speaker_job_rare_label,
    speaker_job_add_length_features=speaker_job_add_length_features,
    speaker_job_add_frequency=speaker_job_add_frequency,
    speaker_job_add_is_rare=speaker_job_add_is_rare,
    speaker_job_add_grouped_job=speaker_job_add_grouped_job,
    speaker_job_add_title_flag=speaker_job_add_title_flag,
    speaker_job_add_comma_flag=speaker_job_add_comma_flag,
    speaker_job_add_slash_flag=speaker_job_add_slash_flag,
    speaker_job_add_ampersand_flag=speaker_job_add_ampersand_flag,
    speaker_job_add_token_count=speaker_job_add_token_count,
    speaker_job_add_job_primary_true_rate=speaker_job_add_job_primary_true_rate,
    speaker_job_job_label_col=speaker_job_job_label_col,
    speaker_job_scale=speaker_job_scale,
    speaker_job_verbose=speaker_job_verbose,

    # --- Party affiliation ---
    party_affiliation_source_col=party_affiliation_source_col,
    party_affiliation_keep_original=party_affiliation_keep_original,
    party_affiliation_clean_text=party_affiliation_clean_text,
    party_affiliation_group_rare=party_affiliation_group_rare,
    party_affiliation_rare_threshold=party_affiliation_rare_threshold,
    party_affiliation_rare_label=party_affiliation_rare_label,
    party_affiliation_add_length_features=party_affiliation_add_length_features,
    party_affiliation_add_frequency=party_affiliation_add_frequency,
    party_affiliation_add_is_rare=party_affiliation_add_is_rare,
    party_affiliation_add_grouped_party=party_affiliation_add_grouped_party,
    party_affiliation_add_slash_flag=party_affiliation_add_slash_flag,
    party_affiliation_add_ampersand_flag=party_affiliation_add_ampersand_flag,
    party_affiliation_add_comma_flag=party_affiliation_add_comma_flag,
    party_affiliation_add_parentheses_flag=party_affiliation_add_parentheses_flag,
    party_affiliation_add_token_count=party_affiliation_add_token_count,
    party_affiliation_add_is_major_party=party_affiliation_add_is_major_party,
    party_affiliation_add_is_institutional=party_affiliation_add_is_institutional,
    party_affiliation_add_party_primary_true_rate=party_affiliation_add_party_primary_true_rate,
    party_affiliation_party_label_col=party_affiliation_party_label_col,
    party_affiliation_scale=party_affiliation_scale,
    party_affiliation_verbose=party_affiliation_verbose,

    # --- State info ---
    state_source_col=state_source_col,
    state_drop=state_drop,
    state_keep_original=state_keep_original,
    state_clean_text=state_clean_text,
    state_normalize_state=state_normalize_state,
    state_group_rare=state_group_rare,
    state_rare_threshold=state_rare_threshold,
    state_rare_label=state_rare_label,
    state_add_is_us_state=state_add_is_us_state,
    state_add_frequency=state_add_frequency,
    state_add_is_rare=state_add_is_rare,
    state_add_grouped_state=state_add_grouped_state,
    state_add_length_features=state_add_length_features,
    state_add_token_count=state_add_token_count,
    state_add_has_us_words=state_add_has_us_words,
    state_add_us_region=state_add_us_region,
    state_scale=state_scale,
    state_verbose=state_verbose,

    # --- Feature engineering ---
    fe_statement_col=fe_statement_col,
    fe_statement_original_col=fe_statement_original_col,
    fe_speaker_col=fe_speaker_col,
    fe_subject_col=fe_subject_col,
    fe_party_col=fe_party_col,
    fe_speaker_job_col=fe_speaker_job_col,
    fe_state_col=fe_state_col,
    fe_label_col=fe_label_col,
    fe_add_speaker_subject=fe_add_speaker_subject,
    fe_add_speaker_party=fe_add_speaker_party,
    fe_add_subject_party=fe_add_subject_party,
    fe_add_speaker_job_subject=fe_add_speaker_job_subject,
    fe_add_state_party=fe_add_state_party,
    fe_add_speaker_statement_len_bucket=fe_add_speaker_statement_len_bucket,
    fe_statement_len_bins=fe_statement_len_bins,
    fe_add_speaker_true_rate=fe_add_speaker_true_rate,
    fe_add_subject_true_rate=fe_add_subject_true_rate,
    fe_add_party_true_rate=fe_add_party_true_rate,
    fe_add_speaker_avg_statement_len=fe_add_speaker_avg_statement_len,
    fe_add_subject_avg_statement_len=fe_add_subject_avg_statement_len,
    fe_add_speaker_avg_punctuation=fe_add_speaker_avg_punctuation,
    fe_add_speaker_avg_number_ratio=fe_add_speaker_avg_number_ratio,
    fe_add_negation_count=fe_add_negation_count,
    fe_add_hedge_count=fe_add_hedge_count,
    fe_add_absolutist_count=fe_add_absolutist_count,
    fe_add_numeral_count=fe_add_numeral_count,
    fe_add_proper_noun_count=fe_add_proper_noun_count,
    fe_add_readability=fe_add_readability,
    fe_add_sentiment=fe_add_sentiment,
    fe_scale=fe_scale,
    fe_verbose=fe_verbose,
)

In [10]:
df_processed = preprocess_one_step(df, options=options)
print(f'Rows: {len(df_processed):,}')
print(f'Features: {df_processed.shape[1]}')
df_processed.head()


Rows: 8,950
Features: 8


,label,statement,subject_clean,statement_clean,speaker_clean,speaker_job_clean,party_affiliation_clean,state_info_clean
0,1,China is in the South China Sea and (building)...,"china,foreign-policy,military",china is in the south china sea and building a...,donald trump,president-elect,republican,new york
1,0,With the resources it takes to execute just ov...,health-care,with the resources it takes to execute just ov...,chris dodd,u.s. senator,democrat,connecticut
2,0,The (Wisconsin) governor has proposed tax give...,"corporations,pundits,taxes,abc-news-week",the wisconsin governor has proposed tax giveaw...,donna brazile,political commentator,democrat,"washington, d.c."
3,1,Says her representation of an ex-boyfriend who...,"candidates-biography,children,ethics,families,...",says her representation of an ex-boyfriend who...,rebecca bradley,unknown,none,unknown
4,0,At protests in Wisconsin against proposed coll...,"health-care,labor,state-budget",at protests in wisconsin against proposed coll...,republican party wisconsin,unknown,republican,wisconsin
